In [1]:
%pip uninstall -y insightface
%pip uninstall -y onnxruntime
%pip install insightface==0.7.3
%pip install onnxruntime
%pip install onnxruntime-silicon

Found existing installation: insightface 0.7.3
Uninstalling insightface-0.7.3:
  Successfully uninstalled insightface-0.7.3
Note: you may need to restart the kernel to use updated packages.
Found existing installation: onnxruntime 1.27.0
Uninstalling onnxruntime-1.27.0:
  Successfully uninstalled onnxruntime-1.27.0
Note: you may need to restart the kernel to use updated packages.
  Using cached insightface-0.7.3-cp313-cp313-macosx_12_0_arm64.whl
Note: you may need to restart the kernel to use updated packages.
  Using cached onnxruntime-1.27.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (5.4 kB)
Using cached onnxruntime-1.27.0-cp313-cp313-macosx_14_0_arm64.whl (18.4 MB)
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement onnxruntime-silicon (from versions: none)
ERROR: No matching distribution found for onnxruntime-silicon
Note: you may need to restart the kernel to use updated packages.


In [2]:
import insightface
import onnxruntime as ort

print(insightface.__version__)
print(ort.__version__)

0.7.3
1.27.0


In [3]:
# imports
import os
import cv2
import shutil
import random

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import pyiqa

from PIL import Image

from retinaface import RetinaFace
from insightface.app import FaceAnalysis

from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# paths
ROOT_DIR = "tinyface_train"

# --------------------------
# Image folders
# --------------------------

GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery",
)

PROBE_DIR = os.path.join(
    ROOT_DIR,
    "probe",
)

GALLERY_CROPPED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_cropped",
)

# --------------------------
# Embedding folders
# --------------------------

GALLERY_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_embeddings",
)

PROBE_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "probe_embeddings",
)

# --------------------------
# Dataset paths
# --------------------------

OUTPUT_DATASET = "tinyface_train_dataset.csv"

OLD_DATASET = "train_mlp_extended_final_version.csv"

FINAL_DATASET = "train_mlp_extended_final_plus_tinyface.csv"

In [15]:
# output directories
os.makedirs(
    GALLERY_CROPPED_DIR,
    exist_ok=True,
)

os.makedirs(
    GALLERY_EMBED_DIR,
    exist_ok=True,
)

os.makedirs(
    PROBE_EMBED_DIR,
    exist_ok=True,
)

print("Directories Ready")

Directories Ready


In [ ]:
# cropping gallery images using retinaFace
failed = 0
cropped = 0

for identity in tqdm(sorted(os.listdir(GALLERY_DIR))):

    identity_dir = os.path.join(
        GALLERY_DIR,
        identity,
    )

    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(
        GALLERY_CROPPED_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    for image_name in sorted(os.listdir(identity_dir)):

        image_path = os.path.join(
            identity_dir,
            image_name,
        )

        img = cv2.imread(image_path)

        if img is None:
            continue

        import time
        print("before Retina face")
        start=time.time()
        faces=RetinaFace.detect_faces(img)
        print("After retina face")
        print("Finished in", time.time()-start)

        if not isinstance(faces, dict) or len(faces)==0:

            failed += 1

            #save original path as fallback
            save_path=os.path.join(save_dir,image_name)
            cv2.imwrite(save_path, img)
            continue

        best_face = max(
            faces.values(),
            key=lambda x:
            (x["facial_area"][2] - x["facial_area"][0]) *
            (x["facial_area"][3] - x["facial_area"][1])
        )

        x1, y1, x2, y2 = best_face["facial_area"]

        h,w=img.shape[:2]
        x1=max(0,x1)
        y1=max(0,y1)
        x2=min(w,x2)
        y2=min(h,y2)
        face=img[y1:y2, x1:x2]
        if face.size==0:
            face=img

        save_path = os.path.join(
            save_dir,
            image_name,
        )

        cv2.imwrite(
            save_path,
            face,
        )

        cropped += 1
        processed=cropped+failed
        if processed % 5 == 0:
            print(f"Processed: {processed} | "
                  f"Cropped: {cropped} | "
                  f"Fallback: {failed}"
            )

saved=cropped+failed
print()
print("Gallery images saved :", saved)
print("successfully cropped :", cropped)
print("Fallback originals :", failed)

  0%|          | 0/95 [00:00<?, ?it/s]

before Retina face


In [16]:
# intializing arcface

app = FaceAnalysis(
    name="buffalo_l",
)

app.prepare(
    ctx_id=0 if torch.cuda.is_available() else -1,
    det_size=(640,640),
)

rec_model = app.models["recognition"]

print("Recognition Model Loaded")
print("ArcFace Loaded")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
se

In [17]:
# --------------------------
# ArcFace Embedding Function
# --------------------------


def get_embedding(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (112, 112))

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    emb = rec_model.get_feat(img).flatten()

    return emb.astype(np.float32)

In [18]:
# --------------------------
# Gallery Embeddings
# --------------------------

saved = 0
failed = 0

for identity in tqdm(sorted(os.listdir(GALLERY_CROPPED_DIR))):

    identity_dir = os.path.join(
        GALLERY_CROPPED_DIR,
        identity,
    )

    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(
        GALLERY_EMBED_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    for image_name in sorted(os.listdir(identity_dir)):

        image_path = os.path.join(
            identity_dir,
            image_name,
        )

        embedding = get_embedding(
            image_path,
        )

        if embedding is None:

            failed += 1
            continue

        save_path = os.path.join(
            save_dir,
            os.path.splitext(image_name)[0] + ".npy",
        )

        np.save(
            save_path,
            embedding,
        )

        saved += 1

print()

print("Gallery Embeddings Saved :", saved)

print("Failures :", failed)

100%|██████████| 95/95 [00:10<00:00,  9.00it/s]


Gallery Embeddings Saved : 95
Failures : 0


In [19]:
identity = sorted(os.listdir(GALLERY_CROPPED_DIR))[0]

image_name = os.listdir(
    os.path.join(GALLERY_CROPPED_DIR, identity)
)[0]

image_path = os.path.join(
    GALLERY_CROPPED_DIR,
    identity,
    image_name,
)

embedding = get_embedding(image_path)

print(type(embedding))
print(embedding.shape)
print(embedding[:10])

<class 'numpy.ndarray'>
(512,)
[-1.254347   -0.30001512  2.0910926   0.66259146 -0.6848353   0.6346671
  0.88206655  0.12475814 -1.8622218  -0.00954206]


In [20]:
# --------------------------
# Probe Embeddings
# --------------------------

saved = 0
failed = 0

for identity in tqdm(sorted(os.listdir(PROBE_DIR))):

    identity_dir = os.path.join(
        PROBE_DIR,
        identity,
    )

    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(
        PROBE_EMBED_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    for image_name in sorted(os.listdir(identity_dir)):

        image_path = os.path.join(
            identity_dir,
            image_name,
        )

        embedding = get_embedding(
            image_path,
        )

        if embedding is None:

            failed += 1
            continue

        save_path = os.path.join(
            save_dir,
            os.path.splitext(image_name)[0] + ".npy",
        )

        np.save(
            save_path,
            embedding,
        )

        saved += 1

print()

print("Probe Embeddings Saved :", saved)

print("Failures :", failed)

100%|██████████| 96/96 [00:09<00:00, 10.25it/s]


Probe Embeddings Saved : 95
Failures : 0


In [21]:
# verifying embeddings
gallery_embeddings = 0

for identity in os.listdir(GALLERY_EMBED_DIR):

    identity_dir = os.path.join(
        GALLERY_EMBED_DIR,
        identity,
    )

    if os.path.isdir(identity_dir):

        gallery_embeddings += len(
            [f for f in os.listdir(identity_dir) if f.endswith(".npy")]
        )

probe_embeddings = 0

for identity in os.listdir(PROBE_EMBED_DIR):

    identity_dir = os.path.join(
        PROBE_EMBED_DIR,
        identity,
    )

    if os.path.isdir(identity_dir):

        probe_embeddings += len(
            [f for f in os.listdir(identity_dir) if f.endswith(".npy")]
        )

print()

print("Gallery Embeddings :", gallery_embeddings)

print("Probe Embeddings   :", probe_embeddings)


Gallery Embeddings : 95
Probe Embeddings   : 95


In [22]:
# --------------------------
# Load MUSIQ
# --------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

musiq_metric = pyiqa.create_metric(
    "musiq",
    device=device,
)

print("MUSIQ Loaded")

Device : cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [23]:
# --------------------------
# MUSIQ Quality Score
# --------------------------


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())

In [24]:
# --------------------------
# Load Gallery Database
# --------------------------


def load_gallery_database(gallery_root):

    gallery_db = {}

    total = 0

    for identity in sorted(os.listdir(gallery_root)):

        identity_dir = os.path.join(
            gallery_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        gallery_db[identity] = []

        for file in sorted(os.listdir(identity_dir)):

            if not file.endswith(".npy"):
                continue

            emb = np.load(
                os.path.join(
                    identity_dir,
                    file,
                )
            )

            gallery_db[identity].append(emb)

            total += 1

    print()

    print("Gallery Identities :", len(gallery_db))
    print("Gallery Embeddings :", total)

    return gallery_db

In [25]:
# --------------------------
# Load Probe Database
# --------------------------


def load_probe_database(probe_root):

    probe_db = {}

    total = 0

    for identity in sorted(os.listdir(probe_root)):

        identity_dir = os.path.join(
            probe_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        files = [f for f in os.listdir(identity_dir) if f.endswith(".npy")]

        if len(files) == 0:
            continue

        emb = np.load(
            os.path.join(
                identity_dir,
                files[0],
            )
        )

        probe_db[identity] = emb

        total += 1

    print()

    print("Probe Embeddings :", total)

    return probe_db

In [26]:
gallery_db = load_gallery_database(
    GALLERY_EMBED_DIR,
)

probe_db = load_probe_database(
    PROBE_EMBED_DIR,
)


Gallery Identities : 95
Gallery Embeddings : 95

Probe Embeddings : 95


In [27]:
# --------------------------
# Gallery Verification
# --------------------------


def search_gallery(
    probe_embedding,
    gallery_db,
):

    identity_scores = {}

    for identity in gallery_db:

        similarities = []

        for gallery_embedding in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1),
                gallery_embedding.reshape(1, -1),
            )[0][0]

            similarities.append(sim)

        identity_scores[identity] = max(similarities)

    sorted_scores = sorted(
        identity_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return (
        identity_scores,
        sorted_scores,
    )

In [28]:
# sanity check
identity = list(probe_db.keys())[0]

identity_scores, sorted_scores = search_gallery(
    probe_db[identity],
    gallery_db,
)

print()

print("True Identity :", identity)

print()

print("Top 5 Matches")

for i in range(5):

    print(sorted_scores[i])


True Identity : Adriana_Lima

Top 5 Matches
('Jessica_Barden', np.float32(0.18951753))
('Rebecca_Ferguson', np.float32(0.16187069))
('Melissa_Fumero', np.float32(0.13678062))
('Amber_Heard', np.float32(0.11709633))
('Gal_Gadot', np.float32(0.099518515))


In [29]:
# ---------------------------------------
# Generate Training Dataset
# ---------------------------------------

rows = []

counter = 0

for identity in tqdm(sorted(probe_db.keys())):

    probe_embedding = probe_db[identity]

    ######################################################
    # locate probe image
    ######################################################

    probe_folder = os.path.join(
        PROBE_DIR,
        identity,
    )

    image_name = None

    for file in os.listdir(probe_folder):

        if file.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
            )
        ):

            image_name = file
            break

    if image_name is None:
        continue

    image_path = os.path.join(
        probe_folder,
        image_name,
    )

    ######################################################
    # MUSIQ
    ######################################################

    quality_score = get_quality_score(
        image_path,
    )

    ######################################################
    # Gallery Search
    ######################################################

    identity_scores, sorted_scores = search_gallery(
        probe_embedding,
        gallery_db,
    )

    ######################################################
    # Genuine Pair
    ######################################################

    genuine_similarity = identity_scores[identity]

    best_impostor_similarity = max(
        score for gallery_id, score in identity_scores.items() if gallery_id != identity
    )

    genuine_margin = genuine_similarity - best_impostor_similarity

    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": genuine_similarity,
            "margin": genuine_margin,
            "label": 1,
            "true_identity": identity,
            "gallery_identity": identity,
        }
    )

    ######################################################
    # Hard Negative
    ######################################################

    impostor_scores = [
        (gallery_id, similarity)
        for gallery_id, similarity in sorted_scores
        if gallery_id != identity
    ]

    hard_negative_identity = impostor_scores[0][0]

    hard_negative_similarity = impostor_scores[0][1]

    second_impostor_similarity = impostor_scores[1][1]

    hard_negative_margin = hard_negative_similarity - second_impostor_similarity

    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": hard_negative_similarity,
            "margin": hard_negative_margin,
            "label": 0,
            "true_identity": identity,
            "gallery_identity": hard_negative_identity,
        }
    )

    counter += 1

    if counter % 25 == 0:

        print()

        print("Processed :", counter)

        print("Quality :", round(quality_score, 2))

        print("Positive Similarity :", round(genuine_similarity, 4))

        print("Negative Similarity :", round(hard_negative_similarity, 4))

 27%|██▋       | 26/95 [00:03<00:08,  8.12it/s]


Processed : 25
Quality : 18.0
Positive Similarity : 0.0577
Negative Similarity : 0.2073


 54%|█████▎    | 51/95 [00:06<00:05,  8.31it/s]


Processed : 50
Quality : 15.06
Positive Similarity : 0.0204
Negative Similarity : 0.1589


 80%|████████  | 76/95 [00:10<00:02,  6.48it/s]


Processed : 75
Quality : 19.39
Positive Similarity : 0.0241
Negative Similarity : 0.1445


100%|██████████| 95/95 [00:13<00:00,  7.03it/s]


In [30]:
# saving the dataset
dataset = pd.DataFrame(rows)

dataset.to_csv(
    OUTPUT_DATASET,
    index=False,
)

print()

print("Dataset Saved")

print(dataset.shape)

display(dataset.head())


Dataset Saved
(190, 6)


,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,19.358395,0.079925,-0.109593,1,Adriana_Lima,Adriana_Lima
1,19.358395,0.189518,0.027647,0,Adriana_Lima,Jessica_Barden
2,19.542381,0.033788,-0.201966,1,Alexandra_Daddario,Alexandra_Daddario
3,19.542381,0.235754,0.056043,0,Alexandra_Daddario,Emma_Stone
4,18.003242,0.240129,0.070674,1,Alvaro_Morte,Alvaro_Morte


In [31]:
# new dataset stats
print()

print("Label Counts")

print(dataset["label"].value_counts())

print()

print("Quality Statistics")

print(dataset["quality_score"].describe())

print()

print("Similarity Statistics")

print(dataset["best_similarity"].describe())

print()

print("Margin Statistics")

print(dataset["margin"].describe())

print()

print("Positive Similarity")

print(dataset[dataset["label"] == 1]["best_similarity"].describe())

print()

print("Negative Similarity")

print(dataset[dataset["label"] == 0]["best_similarity"].describe())

print()

print("Positive Margin")

print(dataset[dataset["label"] == 1]["margin"].describe())

print()

print("Negative Margin")

print(dataset[dataset["label"] == 0]["margin"].describe())


Label Counts
label
1    95
0    95
Name: count, dtype: int64

Quality Statistics
count    190.000000
mean      18.465148
std        2.586939
min       14.313741
25%       16.687280
50%       18.100471
75%       19.721252
max       26.320620
Name: quality_score, dtype: float64

Similarity Statistics
count    190.000000
mean       0.163639
std        0.110260
min       -0.063376
25%        0.086425
50%        0.165773
75%        0.210874
max        0.694695
Name: best_similarity, dtype: float64

Margin Statistics
count    190.000000
mean      -0.039078
std        0.126866
min       -0.664410
25%       -0.110440
50%        0.004380
75%        0.030272
max        0.175305
Name: margin, dtype: float64

Positive Similarity
count    95.000000
mean      0.110307
std       0.090549
min      -0.063376
25%       0.036751
50%       0.086620
75%       0.172274
max       0.347532
Name: best_similarity, dtype: float64

Negative Similarity
count    95.000000
mean      0.216971
std       0.102448
min 

In [32]:
# appending to existing dataset
old_dataset = pd.read_csv(
    OLD_DATASET,
)

print()

print("Old Dataset :", old_dataset.shape)

print("New Dataset :", dataset.shape)

combined_dataset = pd.concat(
    [
        old_dataset,
        dataset,
    ],
    ignore_index=True,
)

combined_dataset = combined_dataset.sample(
    frac=1,
    random_state=42,
).reset_index(
    drop=True,
)

combined_dataset.to_csv(
    FINAL_DATASET,
    index=False,
)

print()

print("Combined Dataset Saved")

print()

print(combined_dataset.shape)


Old Dataset : (4927, 6)
New Dataset : (190, 6)

Combined Dataset Saved

(5117, 6)


In [33]:
# final dataset stats
print()

print("Final Dataset Statistics")

print()

print(combined_dataset["label"].value_counts())

print()

print(combined_dataset.describe())


Final Dataset Statistics

label
1    2582
0    2535
Name: count, dtype: int64

       quality_score  best_similarity       margin        label
count    5117.000000      5117.000000  5117.000000  5117.000000
mean       20.402349         0.427164     0.178972     0.504593
std         5.671775         0.213087     0.207421     0.500028
min        12.476462        -0.063376    -0.664410     0.000000
25%        16.081200         0.246570     0.016324     0.000000
50%        19.092539         0.338243     0.074121     1.000000
75%        23.314095         0.638665     0.375586     1.000000
max        47.028496         0.892248     0.697794     1.000000
